In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("준비 완료!")

준비 완료!


In [2]:
# np.nan = 결측값 (빠진 값)
data = {
    'temp_C':      [200, 250, 300, 350, 400, 450, 500, 550],
    'time_h':      [1,   2,   np.nan, 3, 2,  1,   4,   2  ],
    'hardness_HV': [120, 145, 180, np.nan, 195, 170, 155, 140],
    'tensile_MPa': [350, 410, 490, 530, np.nan, 460, 420, 390],
    'sample_id':   ['S01','S02','S03','S04','S05','S06','S07','S08']
}

df = pd.DataFrame(data)
df

,temp_C,time_h,hardness_HV,tensile_MPa,sample_id
0,200,1.0,120.0,350.0,S01
1,250,2.0,145.0,410.0,S02
2,300,NaN,180.0,490.0,S03
3,350,3.0,NaN,530.0,S04
4,400,2.0,195.0,NaN,S05
5,450,1.0,170.0,460.0,S06
6,500,4.0,155.0,420.0,S07
7,550,2.0,140.0,390.0,S08


In [3]:
print("=== 결측값 개수 ===")
print(df.isnull().sum())

print("\n=== 결측값 위치 ===")
print(df.isnull())

=== 결측값 개수 ===
temp_C         0
time_h         1
hardness_HV    1
tensile_MPa    1
sample_id      0
dtype: int64

=== 결측값 위치 ===
   temp_C  time_h  hardness_HV  tensile_MPa  sample_id
0   False   False        False        False      False
1   False   False        False        False      False
2   False    True        False        False      False
3   False   False         True        False      False
4   False   False        False         True      False
5   False   False        False        False      False
6   False   False        False        False      False
7   False   False        False        False      False


In [4]:
df_clean = df.copy()  # 원본은 보존하고 복사본에서 작업

# 방법 1: 평균값으로 채우기
df_clean['time_h'] = df_clean['time_h'].fillna(df_clean['time_h'].mean())
df_clean['hardness_HV'] = df_clean['hardness_HV'].fillna(df_clean['hardness_HV'].mean())
df_clean['tensile_MPa'] = df_clean['tensile_MPa'].fillna(df_clean['tensile_MPa'].mean())

print("=== 처리 후 결측값 개수 ===")
print(df_clean.isnull().sum())

print("\n=== 채워진 값 확인 ===")
print(df_clean)

=== 처리 후 결측값 개수 ===
temp_C         0
time_h         0
hardness_HV    0
tensile_MPa    0
sample_id      0
dtype: int64

=== 채워진 값 확인 ===
   temp_C    time_h  hardness_HV  tensile_MPa sample_id
0     200  1.000000   120.000000   350.000000       S01
1     250  2.000000   145.000000   410.000000       S02
2     300  2.142857   180.000000   490.000000       S03
3     350  3.000000   157.857143   530.000000       S04
4     400  2.000000   195.000000   435.714286       S05
5     450  1.000000   170.000000   460.000000       S06
6     500  4.000000   155.000000   420.000000       S07
7     550  2.000000   140.000000   390.000000       S08


In [5]:
# 온도 구간 나누기
df_clean['temp_range'] = pd.cut(df_clean['temp_C'],
                                bins=[0, 300, 450, 600],
                                labels=['low', 'mid', 'high'])

# 구간별 평균 계산
grouped = df_clean.groupby('temp_range')[['hardness_HV', 'tensile_MPa']].mean()
print("=== 온도 구간별 평균 ===")
print(grouped)

# 구간별 최대값
print("\n=== 온도 구간별 최대값 ===")
print(df_clean.groupby('temp_range')[['hardness_HV', 'tensile_MPa']].max())

=== 온도 구간별 평균 ===
            hardness_HV  tensile_MPa
temp_range                          
low          148.333333   416.666667
mid          174.285714   475.238095
high         147.500000   405.000000

=== 온도 구간별 최대값 ===
            hardness_HV  tensile_MPa
temp_range                          
low               180.0        490.0
mid               195.0        530.0
high              155.0        420.0


In [6]:
# 새 열 추가: 강도/경도 비율
df_clean['strength_ratio'] = df_clean['tensile_MPa'] / df_clean['hardness_HV']

# 경도 기준으로 등급 분류
def classify_grade(hv):
    if hv >= 180:
        return 'A'
    elif hv >= 150:
        return 'B'
    else:
        return 'C'

df_clean['grade'] = df_clean['hardness_HV'].apply(classify_grade)

print("=== 새 열 추가 결과 ===")
print(df_clean[['sample_id', 'hardness_HV', 'tensile_MPa', 'strength_ratio', 'grade']])

print("\n=== 등급별 샘플 수 ===")
print(df_clean['grade'].value_counts())

=== 새 열 추가 결과 ===
  sample_id  hardness_HV  tensile_MPa  strength_ratio grade
0       S01   120.000000   350.000000        2.916667     C
1       S02   145.000000   410.000000        2.827586     C
2       S03   180.000000   490.000000        2.722222     A
3       S04   157.857143   530.000000        3.357466     B
4       S05   195.000000   435.714286        2.234432     A
5       S06   170.000000   460.000000        2.705882     B
6       S07   155.000000   420.000000        2.709677     B
7       S08   140.000000   390.000000        2.785714     C

=== 등급별 샘플 수 ===
grade
C    3
B    3
A    2
Name: count, dtype: int64
